# glassbox — Phase 3, TinyStories

The first run on a corpus that does not fit in memory, with a model large
enough to write coherent English. Roughly **11M parameters**, a 4,096-token BPE
vocabulary, and a 256-token context.

Two stages. **Preparation** downloads 1.9 GB, learns the merges and writes token
files — run once, and it lands in Drive so a disconnect does not repeat it.
**Training** memory-maps those files, so nothing is tokenised while the GPU is
waiting.

If the session drops mid-training, set `RESUME = True` and rerun the training
cell. It continues from `last.pt` with the optimizer moments intact.

## 1 · Hardware

In [1]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no GPU visible")

import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"device       {torch.cuda.get_device_name(0)}")
    print(f"capability   {major}.{minor}  "
          f"({'Ampere or newer' if major >= 8 else 'pre-Ampere'})")
    print(f"bf16 native  {major >= 8}")
else:
    print("No GPU. Runtime > Change runtime type > GPU.")

Tesla T4, 15360 MiB

device       Tesla T4
capability   7.5  (pre-Ampere)
bf16 native  False


## 2 · Clone, install, mount

In [3]:
import os, subprocess

ROOT = "/content/glassbox"
if os.path.isdir(ROOT):
    print(subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"],
                         capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/udit-rawat/glassbox.git", ROOT], check=True)

os.chdir(ROOT)
subprocess.run(["pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATA = "/content/drive/MyDrive/glassbox/data/tinystories"
LOCAL_DATA = "/content/data/tinystories"
OUT_DIR    = "/content/drive/MyDrive/glassbox/tinystories"
os.makedirs(OUT_DIR, exist_ok=True)

from glassbox.device import get_device
from glassbox.training.precision import select_precision
p = select_precision(get_device(), enabled=True)
print(f"precision   {p.describe()}   scaler {p.use_scaler}")

Mounted at /content/drive
precision   float16 + scaler   scaler True


## 3 · Prepare the corpus

Downloads TinyStories, learns 3,840 merges from a 40 MB sample, and encodes the
chosen slice into flat `uint16` files.

Merges are learned from a sample rather than the whole corpus on purpose:
TinyStories uses a deliberately small vocabulary, so 40 MB already contains
almost every word, and more text would multiply the cost for merges that come
out nearly identical.

**Skip this cell entirely on a rerun** — it detects the files in Drive and
returns immediately.

In [4]:
TRAIN_MB   = 500     # of the 1.9 GB corpus
VOCAB_SIZE = 4096
BPE_MB     = 40

prepared = all(os.path.exists(f"{DRIVE_DATA}/{f}")
               for f in ("train.bin", "val.bin", "tokenizer.json"))

if prepared:
    print(f"already prepared in {DRIVE_DATA} — skipping")
else:
    cmd = ["python", "-u", "scripts/prepare_tinystories.py",
           "--data-dir", "/content/data/raw",
           "--out-dir", DRIVE_DATA,
           "--train-mb", str(TRAIN_MB),
           "--vocab-size", str(VOCAB_SIZE),
           "--bpe-mb", str(BPE_MB)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

downloading (1.9 GB on the first run, cached after)
  train  1.92 GB
  valid  19.4 MB

learning 3840 merges from 40 MB
  merge   500  freq   7,465
  merge  1000  freq   2,692
  merge  1500  freq   1,380
  merge  2000  freq     783
  merge  2500  freq     510
  merge  3000  freq     374
  merge  3500  freq     292
  vocab 4096 in 15.6s

encoding
  train:  1.6%     2,120,019 tokens     1.1s
  train:  3.2%     4,245,249 tokens     2.1s
  train:  4.8%     6,359,419 tokens     3.0s
  train:  6.4%     8,480,067 tokens     3.9s
  train:  8.0%    10,593,202 tokens     4.9s
  train:  9.6%    12,716,099 tokens     5.8s
  train: 11.2%    14,840,930 tokens     6.7s
  train: 12.8%    16,960,592 tokens     7.6s
  train: 14.4%    19,073,375 tokens     8.6s
  train: 16.0%    21,179,683 tokens     9.5s
  train: 17.6%    23,300,153 tokens    10.9s
  train: 19.2%    25,437,535 tokens    12.6s
  train: 20.8%    27,562,666 tokens    14.0s
  train: 22.4%    29,691,753 tokens    14.9s
  train: 24.0%    31,82

## 4 · Copy the token files to local disk

Drive is a network mount. Memory-mapping across it means every batch waits on a
round trip, which starves the GPU exactly as tokenising in Python would. The
files stay in Drive as the durable copy; training reads a local one.

In [5]:
import shutil, time

os.makedirs(LOCAL_DATA, exist_ok=True)
for f in ("train.bin", "val.bin", "tokenizer.json"):
    dst = f"{LOCAL_DATA}/{f}"
    if not os.path.exists(dst):
        t0 = time.time()
        shutil.copy(f"{DRIVE_DATA}/{f}", dst)
        print(f"{f:<16} {os.path.getsize(dst)/1e6:>8.1f} MB  {time.time()-t0:5.1f}s")
    else:
        print(f"{f:<16} {os.path.getsize(dst)/1e6:>8.1f} MB  (already local)")

train.bin           265.8 MB    0.5s
val.bin               9.8 MB    0.0s
tokenizer.json        0.1 MB    0.0s


## 5 · Sanity check

In [6]:
print(subprocess.run(["python", "-m", "pytest", "tests/", "-q", "--no-header"],
                     capture_output=True, text=True).stdout[-800:])

........................................................................ [ 61%]
.............................................                            [100%]
117 passed in 11.63s



## 6 · Train

Mixed precision is on here, unlike the Phase 1 restore — there is no old number
to reproduce, so speed wins. The header prints how many tokens the run will
consume against how many exist; above 1.0 epochs the model is seeing the corpus
more than once.

Watch the first two evaluations. Loss should start near `ln(4096) ≈ 8.3` and
fall quickly. If it sits flat, stop and check rather than waiting hours.

In [7]:
MAX_ITERS  = 20000
BATCH_SIZE = 32
GRAD_ACCUM = 1
LR         = 6e-4
RESUME     = False   # True after a disconnect

cmd = ["python", "-u", "scripts/train_tinystories.py",
       "--data-dir", LOCAL_DATA,
       "--out-dir", OUT_DIR,
       "--max-iters", str(MAX_ITERS),
       "--batch-size", str(BATCH_SIZE),
       "--grad-accum", str(GRAD_ACCUM),
       "--lr", str(LR),
       "--schedule", "cosine",
       "--eval-interval", "500",
       "--sample-tokens", "300"]
if RESUME:
    cmd.append("--resume")

print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

python -u scripts/train_tinystories.py --data-dir /content/data/tinystories --out-dir /content/drive/MyDrive/glassbox/tinystories --max-iters 20000 --batch-size 32 --grad-accum 1 --lr 0.0006 --schedule cosine --eval-interval 500 --sample-tokens 300 

device      cuda
data        TokenDataset(train=132,881,504 tokens, val=4,903,303 tokens)
vocab       4096
arch        rmsnorm / swiglu / rope / kv_heads 2
parameters  11,015,040
tokens      163,840,000 to be seen (1.23 epochs)
out_dir     /content/drive/MyDrive/glassbox/tinystories

precision   float16 + scaler   effective batch 32 (32 x 1)   schedule cosine
iter    500  train 2.8910  val 2.8883  lr 6.00e-04     70.1s
iter   2000  train 2.1039  val 2.1143  lr 5.92e-04    289.4s
iter   2500  train 2.0308  val 2.0478  lr 5.86e-04    362.8s
iter   3000  train 1.9812  val 1.9849  lr 5.78e-04    435.9s
iter   3500  train 1.9361  val 1.9564  lr 5.69e-04    509.0s
iter   4000  train 1.9148  val 1.9313  lr 5.58e-04    582.2s
iter   4500  train 1.

0

## 7 · Sample

The real test of Phase 3: does it write English? Nucleus sampling rather than
top-k, because the candidate set should widen where the story could go several
ways and narrow mid-word.

In [8]:
for prompt in ["Once upon a time, there was a little",
               "Tom and Sara went to the park. They saw",
               "The dog was very sad because"]:
    print("=" * 70)
    print(subprocess.run([
        "python", "scripts/sample.py",
        "--checkpoint", f"{OUT_DIR}/best.pt",
        "--prompt", prompt,
        "--tokens", "200",
        "--temperature", "0.8",
        "--top-p", "0.9",
    ], capture_output=True, text=True).stdout)

checkpoint  iter 20000, val loss 1.5787
arch        rmsnorm / swiglu / rope / kv_heads 2
tokenizer   BPETokenizer, vocab 4096
sampling    temp 0.8, top_k None, top_p 0.9

Once upon a time, there was a little boy named Tim. Tim had a very big toy box. He loved to play with his toys every day. One day, he went to the park to play with his toys.
While playing, Tim met a new friend named Sam. Sam wanted to play with the big toy box too. They both wanted to play with the big toy box. Tim said, "I want to play with the big toy box." Sam said, "No, it's mine!"
Tim and Sam started to fight. They both wanted the big toy box. They pulled and pulled. The big toy box did not want to share. Tim and Sam started to fight over the big toy box. They pulled and pulled until the big toy box broke. The big toy box was now empty.
In the end, the big toy box broke and Tim and Sam felt sad. They both learned that fighting can be not fun. The moral of the story is to not fight and be kind to each other.


che

## 8 · What came back

In [9]:
import math, torch
from pathlib import Path

for f in sorted(Path(OUT_DIR).iterdir()):
    print(f"{f.name:<20} {f.stat().st_size / 1e6:>8.1f} MB")

ckpt = torch.load(f"{OUT_DIR}/best.pt", map_location="cpu", weights_only=False)
cfg = ckpt["model_config"]
loss = ckpt["val_loss"]
print(f"\niter        {ckpt['iter']:,}")
print(f"val loss    {loss:.4f}")
# Perplexity reads as an effective number of choices per token. Starting from
# ln(vocab), anything near single digits means the model has real structure.
print(f"perplexity  {math.exp(loss):.1f}   (uniform would be {cfg.vocab_size})")
print(f"arch        {cfg.norm} / {cfg.activation} / {cfg.pos_encoding} / kv={cfg.n_kv_heads}")
print(f"tokenizer   {ckpt['tokenizer']['kind']}, {len(ckpt['tokenizer']['merges'])} merges")

best.pt                 132.3 MB
last.pt                 132.3 MB

iter        20,000
val loss    1.5787
perplexity  4.8   (uniform would be 4096)
arch        rmsnorm / swiglu / rope / kv=2
tokenizer   bpe, 3840 merges


## 9 · Export a slim checkpoint

The training checkpoint is 132 MB because it carries AdamW's two moment
tensors per parameter so a dropped run can resume. Inference never reads them,
and they are two thirds of the file.

This writes a version with the optimizer state removed — everything needed to
rebuild and run the model, nothing needed to continue training it. That is the
file worth downloading for the visualizer.


In [ ]:
print(subprocess.run([
    "python", "scripts/export_checkpoint.py",
    f"{OUT_DIR}/best.pt",
    "--out", f"{OUT_DIR}/tinystories_11m.pt",
], capture_output=True, text=True).stdout)

# Half precision halves it again, to roughly 22 MB. Fine for a demo; not what
# you want if this model will be fine-tuned later, because the optimizer would
# inherit the rounding.
# ... add --half to the command above
